## Partitions, hot keys & capacity modes

Under the hood DynamoDB shards data across many physical **partitions**, chosen by **hashing the partition key** — and two implications bite in production. First, **you want high cardinality on the partition key**: if millions of items share a few PK values (`status="active"`, `country="US"`), almost all traffic hits **one physical partition**, whose throughput is capped, and the table **throttles even when total capacity looks healthy** — the classic **hot partition**. Fix it by choosing a high-cardinality key or synthesising one (e.g. `status#shardId`). Second, **you can't query across partitions cheaply** — a `Query` always targets one PK value; needing many PKs points at a **GSI** (next section).

**Capacity modes** shape cost and behaviour. **Provisioned** — you set **RCUs/WCUs** upfront (1 RCU = one strongly-consistent 4 KB read/s, or two eventually-consistent; 1 WCU = one 1 KB write/s), cheaper for **steady/predictable** load but **throttles** past the ceiling; **Auto Scaling** adjusts within a min/max on a minute scale. **On-demand** — no planning, scales instantly, pay per request, **never throttles on spikes**, costs more per request — right for **spiky or unknown** workloads.

The heuristic: **on-demand for new/spiky/unknown, provisioned + Auto Scaling for steady high-volume** where you can plan.